# 02 数据清洗

本Notebook根据原始数据质量审计结果，完成重复记录处理、字段格式标准化、交易类型分类、特殊商品编码分类和清洗结果验证。

原始数据只读取，不覆盖。

In [3]:
from pathlib import Path
import pandas as pd

In [4]:
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

raw_path = (
    project_root
    / "data"
    / "raw"
    / "Online Retail.xlsx"
)

processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

assert raw_path.exists(), f"找不到原始数据：{raw_path}"

df_raw = pd.read_excel(
    raw_path,
    sheet_name="Online Retail"
)

raw_columns = df_raw.columns.tolist()

print("项目根目录：", project_root)
print("原始数据路径：", raw_path)
print("原始数据形状：", df_raw.shape)

项目根目录： c:\Users\a\Documents\DataAnalysisProjects\ecommerce-customer-analysis
原始数据路径： c:\Users\a\Documents\DataAnalysisProjects\ecommerce-customer-analysis\data\raw\Online Retail.xlsx
原始数据形状： (541909, 8)


In [5]:
df_clean = df_raw.copy()

# 记录原始Excel中的行号。
# pandas索引0对应Excel第2行，因此加2。
df_clean.insert(
    0,
    "SourceRow",
    df_clean.index + 2
)

duplicate_mask = df_clean.duplicated(
    subset=raw_columns,
    keep="first"
)

duplicate_count = int(duplicate_mask.sum())

print("准备删除的完全重复行：", duplicate_count)

df_clean = (
    df_clean.loc[~duplicate_mask]
    .copy()
    .reset_index(drop=True)
)

print("删除重复记录后的行数：", len(df_clean))

assert duplicate_count == 5268
assert len(df_clean) == 536641
assert df_clean.duplicated(
    subset=raw_columns
).sum() == 0

print("重复记录处理验证通过")

准备删除的完全重复行： 5268
删除重复记录后的行数： 536641
重复记录处理验证通过


In [6]:
df_clean["InvoiceNo"] = (
    df_clean["InvoiceNo"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_clean["StockCode"] = (
    df_clean["StockCode"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_clean["Description"] = (
    df_clean["Description"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

df_clean["CustomerID"] = (
    df_clean["CustomerID"]
    .astype("Int64")
    .astype("string")
)

df_clean["Country"] = (
    df_clean["Country"]
    .astype("string")
    .str.strip()
)

df_clean["InvoiceDate"] = pd.to_datetime(
    df_clean["InvoiceDate"],
    errors="raise"
)

df_clean.dtypes

SourceRow               int64
InvoiceNo              string
StockCode              string
Description            string
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID             string
Country                string
dtype: object

In [7]:
df_clean["LineAmount"] = (
    df_clean["Quantity"]
    * df_clean["UnitPrice"]
)

df_clean["Year"] = df_clean["InvoiceDate"].dt.year

df_clean["Month"] = (
    df_clean["InvoiceDate"]
    .dt.to_period("M")
    .astype("string")
)

df_clean["HasCustomerID"] = (
    df_clean["CustomerID"].notna()
)

is_cancelled = (
    df_clean["InvoiceNo"]
    .str.startswith("C", na=False)
)

is_stock_adjustment = (
    (df_clean["Quantity"] < 0)
    & ~is_cancelled
)

is_price_adjustment = (
    (df_clean["UnitPrice"] < 0)
    & ~is_cancelled
    & (df_clean["Quantity"] >= 0)
)

is_zero_price = (
    (df_clean["UnitPrice"] == 0)
    & ~is_cancelled
    & (df_clean["Quantity"] > 0)
)

is_sale = (
    (df_clean["Quantity"] > 0)
    & (df_clean["UnitPrice"] > 0)
    & ~is_cancelled
)

df_clean["TransactionType"] = "Other"

df_clean.loc[
    is_cancelled,
    "TransactionType"
] = "Cancellation"

df_clean.loc[
    is_stock_adjustment,
    "TransactionType"
] = "StockAdjustment"

df_clean.loc[
    is_price_adjustment,
    "TransactionType"
] = "PriceAdjustment"

df_clean.loc[
    is_zero_price,
    "TransactionType"
] = "ZeroPrice"

df_clean.loc[
    is_sale,
    "TransactionType"
] = "Sale"

df_clean["TransactionType"].value_counts()

TransactionType
Sale               524878
Cancellation         9251
StockAdjustment      1336
ZeroPrice            1174
PriceAdjustment         2
Name: count, dtype: int64

In [8]:
special_code_map = {
    "POST": "Postage",
    "DOT": "Postage",
    "M": "Manual",
    "C2": "Carriage",
    "D": "Discount",
    "S": "Sample",
    "BANK CHARGES": "BankCharge",
    "AMAZONFEE": "PlatformFee",
    "CRUK": "Commission",
    "B": "BadDebtAdjustment"
}

df_clean["ItemType"] = (
    df_clean["StockCode"]
    .map(special_code_map)
    .fillna("Merchandise")
    .astype("string")
)

gift_voucher_mask = (
    df_clean["StockCode"]
    .str.startswith("GIFT_", na=False)
)

df_clean.loc[
    gift_voucher_mask,
    "ItemType"
] = "GiftVoucher"

df_clean["ItemType"].value_counts()

ItemType
Merchandise          533701
Postage                1966
Manual                  567
Carriage                144
Discount                 77
Sample                   62
BankCharge               37
PlatformFee              34
GiftVoucher              34
Commission               16
BadDebtAdjustment         3
Name: count, dtype: Int64

In [9]:
df_sales = df_clean.loc[
    df_clean["TransactionType"] == "Sale"
].copy()

df_customer_sales = df_sales.loc[
    df_sales["HasCustomerID"]
].copy()

cleaning_summary = pd.Series({
    "原始数据行数": len(df_raw),
    "删除的完全重复行": duplicate_count,
    "清洗后总行数": len(df_clean),
    "正常销售行数": len(df_sales),
    "有客户编号的正常销售行数": len(df_customer_sales),
    "正常销售客户数": (
        df_customer_sales["CustomerID"].nunique()
    ),
    "清洗后CustomerID缺失": (
        df_clean["CustomerID"].isna().sum()
    ),
    "清洗后Description缺失": (
        df_clean["Description"].isna().sum()
    )
})

cleaning_summary

原始数据行数              541909
删除的完全重复行              5268
清洗后总行数              536641
正常销售行数              524878
有客户编号的正常销售行数        392692
正常销售客户数               4338
清洗后CustomerID缺失     135037
清洗后Description缺失      1454
dtype: int64

In [10]:
expected_transaction_counts = {
    "Sale": 524878,
    "Cancellation": 9251,
    "StockAdjustment": 1336,
    "ZeroPrice": 1174,
    "PriceAdjustment": 2
}

actual_transaction_counts = (
    df_clean["TransactionType"]
    .value_counts()
    .to_dict()
)

assert df_raw.shape == (541909, 8)
assert len(df_clean) == 536641
assert df_clean["SourceRow"].is_unique
assert actual_transaction_counts == expected_transaction_counts

assert df_clean["CustomerID"].isna().sum() == 135037
assert df_clean["Description"].isna().sum() == 1454

assert (
    df_clean["CustomerID"]
    .dropna()
    .str.endswith(".0")
    .sum()
    == 0
)

assert len(df_sales) == 524878
assert len(df_customer_sales) == 392692

assert (
    df_customer_sales["CustomerID"].nunique()
    == 4338
)

assert (
    df_sales["Quantity"] > 0
).all()

assert (
    df_sales["UnitPrice"] > 0
).all()

assert (
    ~df_sales["InvoiceNo"]
    .str.startswith("C", na=False)
).all()

assert (
    df_clean["TransactionType"] != "Other"
).all()

assert df_clean["LineAmount"].notna().all()

print("全部清洗验证通过")

全部清洗验证通过


In [11]:
full_output_path = (
    processed_dir
    / "transactions_clean.csv"
)

sample_output_path = (
    processed_dir
    / "transactions_clean_sample.csv"
)

df_clean.to_csv(
    full_output_path,
    index=False,
    encoding="utf-8-sig"
)

sample_df = (
    df_clean
    .sample(n=1000, random_state=42)
    .sort_values("SourceRow")
)

sample_df.to_csv(
    sample_output_path,
    index=False,
    encoding="utf-8-sig"
)

check_df = pd.read_csv(
    full_output_path,
    usecols=[
        "SourceRow",
        "InvoiceNo",
        "StockCode",
        "CustomerID",
        "TransactionType"
    ],
    dtype={
        "InvoiceNo": "string",
        "StockCode": "string",
        "CustomerID": "string",
        "TransactionType": "string"
    }
)

assert len(check_df) == len(df_clean)

print("完整数据：", full_output_path)
print("完整数据行数：", len(check_df))
print("GitHub样本：", sample_output_path)
print("样本行数：", len(sample_df))
print("CSV重新读取验证通过")

完整数据： c:\Users\a\Documents\DataAnalysisProjects\ecommerce-customer-analysis\data\processed\transactions_clean.csv
完整数据行数： 536641
GitHub样本： c:\Users\a\Documents\DataAnalysisProjects\ecommerce-customer-analysis\data\processed\transactions_clean_sample.csv
样本行数： 1000
CSV重新读取验证通过
